In [18]:
import pandas as pd, numpy as np, json

df = pd.read_csv("../data/raw/USvideos.csv")
print(df.shape)
df.head()

(40949, 16)


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description
0,2kyS6SvSYSE,17.14.11,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,2017-11-13T17:13:01.000Z,SHANtell martin,748374,57527,2966,15954,https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg,False,False,False,SHANTELL'S CHANNEL - https://www.youtube.com/shantellmartin\nCANDICE - https://www.lovebilly.com...
1,1ZAPwfrtAFY,17.14.11,The Trump Presidency: Last Week Tonight with John Oliver (HBO),LastWeekTonight,24,2017-11-13T07:30:00.000Z,"last week tonight trump presidency|""last week tonight donald trump""|""john oliver trump""|""donald ...",2418783,97185,6146,12703,https://i.ytimg.com/vi/1ZAPwfrtAFY/default.jpg,False,False,False,"One year after the presidential election, John Oliver discusses what we've learned so far and en..."
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Lele Pons",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""|""racist""|""superman""|""love""|""rudy mancuso poo bear...",3191434,146033,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO ▶ \n\nSUBSCRIBE ► https://www.youtube.com/channel/UC5jkXpfnBhlDjqh0ir5Fs...
3,puqaWrEC7tY,17.14.11,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,24,2017-11-13T11:00:04.000Z,"rhett and link|""gmm""|""good mythical morning""|""rhett and link good mythical morning""|""good mythic...",343168,10172,666,2146,https://i.ytimg.com/vi/puqaWrEC7tY/default.jpg,False,False,False,Today we find out if Link is a Nickelback amateur or a secret Nickelback devotee. GMM #1218\nDon...
4,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12T18:01:41.000Z,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""idy""|""rhpc""|""dares""|""no truth""|""comments""|""comedy""...",2095731,132235,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,"I know it's been a while since we did this show, but we're back with what might be the best epis..."


In [19]:
df.info()
print(df.isna().sum())
print(df.duplicated("video_id").sum(), "repeat rows for the same video")

<class 'pandas.DataFrame'>
RangeIndex: 40949 entries, 0 to 40948
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   video_id                40949 non-null  str  
 1   trending_date           40949 non-null  str  
 2   title                   40949 non-null  str  
 3   channel_title           40949 non-null  str  
 4   category_id             40949 non-null  int64
 5   publish_time            40949 non-null  str  
 6   tags                    40949 non-null  str  
 7   views                   40949 non-null  int64
 8   likes                   40949 non-null  int64
 9   dislikes                40949 non-null  int64
 10  comment_count           40949 non-null  int64
 11  thumbnail_link          40949 non-null  str  
 12  comments_disabled       40949 non-null  bool 
 13  ratings_disabled        40949 non-null  bool 
 14  video_error_or_removed  40949 non-null  bool 
 15  description             40379 

In [20]:
# Make sure the number columns are really numbers
for c in ["views", "likes", "comment_count"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# Flag deleted videos
df["is_deleted"] = df["video_error_or_removed"].astype(str).str.lower().eq("true")

# Flag videos with zero views
df["is_zero_views"] = df["views"] == 0

print("Deleted:", df["is_deleted"].sum())
print("Zero views:", df["is_zero_views"].sum())

Deleted: 23
Zero views: 0


In [21]:
df["engagement_rate"] = np.where(
    df["views"] > 0,
    (df["likes"] + df["comment_count"]) / df["views"].replace(0, np.nan) * 100,
    np.nan,
).round(4)

df[["title", "views", "likes", "comment_count", "engagement_rate"]].head()

,title,views,likes,comment_count,engagement_rate
0,WE WANT TO TALK ABOUT OUR MARRIAGE,748374,57527,15954,9.8188
1,The Trump Presidency: Last Week Tonight with John Oliver (HBO),2418783,97185,12703,4.5431
2,"Racist Superman | Rudy Mancuso, King Bach & Lele Pons",3191434,146033,8181,4.8321
3,Nickelback Lyrics: Real or Fake?,343168,10172,2146,3.5895
4,I Dare You: GOING BALD!?,2095731,132235,17518,7.1456


In [22]:
df["publish_time"] = pd.to_datetime(df["publish_time"], utc=True)
df["publish_time_ist"] = df["publish_time"].dt.tz_convert("Asia/Kolkata")
df["post_date"] = df["publish_time_ist"].dt.date
df["post_day"] = df["publish_time_ist"].dt.day_name()
df["post_hour"] = df["publish_time_ist"].dt.hour

df[["title", "post_date", "post_day", "post_hour"]].head()

,title,post_date,post_day,post_hour
0,WE WANT TO TALK ABOUT OUR MARRIAGE,2017-11-13,Monday,22
1,The Trump Presidency: Last Week Tonight with John Oliver (HBO),2017-11-13,Monday,13
2,"Racist Superman | Rudy Mancuso, King Bach & Lele Pons",2017-11-13,Monday,0
3,Nickelback Lyrics: Real or Fake?,2017-11-13,Monday,16
4,I Dare You: GOING BALD!?,2017-11-12,Sunday,23


In [23]:
q1, q3 = df["views"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
df["is_outlier"] = df["views"] > upper
print("Outlier videos:", df["is_outlier"].sum(), "| cutoff:", int(upper), "views")

Outlier videos: 4499 | cutoff: 4194399 views


In [24]:
def content_type(title):
    t = str(title).lower()
    rules = [
        ("Highlights", ["highlight", "goals", "recap"]),
        ("Trailer / Teaser", ["trailer", "teaser", "first look"]),
        ("Music Video", ["official video", "music video", "lyrics", "song"]),
        ("Interview", ["interview", "press conference", "talks"]),
        ("Challenge / Comedy", ["challenge", "prank", "funny", "comedy", "sketch"]),
        ("Talk Show / Clip", ["show", "monologue", "clip", "tonight"]),
        ("Behind the Scenes", ["behind the scenes", "making of", "vlog"]),
    ]
    for name, words in rules:
        if any(w in t for w in words):
            return name
    return "Other"

df["content_type"] = df["title"].apply(content_type)
df["content_type"].value_counts()

content_type
Other                 33603
Trailer / Teaser       2443
Music Video            2148
Talk Show / Clip        840
Challenge / Comedy      818
Highlights              412
Interview               354
Behind the Scenes       331
Name: count, dtype: int64

In [25]:
df[["title", "content_type"]].sample(20, random_state=1)

,title,content_type
22745,How Good is McDonald's New Szechuan Sauce?,Other
33187,Cobra Kai Ep 2 - Strike First - The Karate Kid Saga Continues,Other
12833,U2 - Get Out Of Your Own Way (Official Video),Music Video
25194,Jurassic Park Auditions - SNL,Other
7802,SZA - The Weekend (Official Video),Music Video
8110,"Prince Harry, Meghan Markle Release Engagement Photos",Other
5164,i was so uncomfortable,Other
24840,See through Rotary Engine in Slow Motion - (Wankel Engine) 4K,Other
12997,Marvel Studios' Black Panther - Warriors of Wakanda,Other
21698,Exercise Ball Magnus Effect from 200m!!,Other


In [26]:
import pandas as pd, numpy as np, json

# 1. Load fresh
df = pd.read_csv("../data/raw/USvideos.csv")
raw_rows = len(df)

# 2. Category names
with open("../data/raw/US_category_id.json") as f:
    cats = json.load(f)
cat_map = {int(i["id"]): i["snippet"]["title"] for i in cats["items"]}
df["category"] = df["category_id"].map(cat_map)

# 3. Keep only Sports and Entertainment
df = df[df["category"].isin(["Sports", "Entertainment"])]

# 4. One row per video (keep highest views)
df = df.sort_values("views", ascending=False).drop_duplicates("video_id")

# 5. Fix types and flags
for c in ["views", "likes", "comment_count"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
df["is_deleted"] = df["video_error_or_removed"].astype(str).str.lower().eq("true")
df["is_zero_views"] = df["views"] == 0

# 6. Engagement rate
df["engagement_rate"] = np.where(
    df["views"] > 0,
    (df["likes"] + df["comment_count"]) / df["views"].replace(0, np.nan) * 100,
    np.nan,
).round(4)

# 7. Day and hour in IST
df["publish_time"] = pd.to_datetime(df["publish_time"], utc=True)
df["publish_time_ist"] = df["publish_time"].dt.tz_convert("Asia/Kolkata")
df["post_date"] = df["publish_time_ist"].dt.date
df["post_day"] = df["publish_time_ist"].dt.day_name()
df["post_hour"] = df["publish_time_ist"].dt.hour

# 8. Outliers
q1, q3 = df["views"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
df["is_outlier"] = df["views"] > upper

# 9. Content type
def content_type(title):
    t = str(title).lower()
    rules = [
        ("Highlights", ["highlight", "goals", "recap"]),
        ("Trailer / Teaser", ["trailer", "teaser", "first look"]),
        ("Music Video", ["official video", "music video", "lyrics", "song"]),
        ("Interview", ["interview", "press conference", "talks"]),
        ("Challenge / Comedy", ["challenge", "prank", "funny", "comedy", "sketch"]),
        ("Talk Show / Clip", ["show", "monologue", "clip", "tonight"]),
        ("Behind the Scenes", ["behind the scenes", "making of", "vlog"]),
    ]
    for name, words in rules:
        if any(w in t for w in words):
            return name
    return "Other"
df["content_type"] = df["title"].apply(content_type)

# 10. Summary
print("Rows at start:", raw_rows, "| Rows after cleaning:", len(df))
print(df["category"].value_counts())
print("Deleted:", df.is_deleted.sum(), "| Zero views:", df.is_zero_views.sum(), "| Outliers:", df.is_outlier.sum())
print(df["content_type"].value_counts())

Rows at start: 40949 | Rows after cleaning: 2073
category
Entertainment    1622
Sports            451
Name: count, dtype: int64
Deleted: 2 | Zero views: 0 | Outliers: 225
content_type
Other                 1634
Trailer / Teaser       174
Talk Show / Clip        77
Highlights              73
Challenge / Comedy      45
Interview               33
Music Video             22
Behind the Scenes       15
Name: count, dtype: int64


In [27]:
out_cols = ["video_id", "title", "channel_title", "category", "content_type",
            "publish_time", "post_date", "post_day", "post_hour", "views", "likes",
            "comment_count", "engagement_rate", "is_deleted", "is_zero_views", "is_outlier"]
df[out_cols].to_csv("../data/clean/posts_clean.csv", index=False)
print("Saved", len(df), "rows")

Saved 2073 rows


In [28]:
pd.set_option("display.max_colwidth", 100)
df[df["content_type"] == "Other"]["title"].sample(40, random_state=2)

33300                  Amy Schumer Shares Her Experience with Gray-Area Rape | SuperSoul Conversations | OWN
36770                                                   James Corden Explains His Resting Royal Wedding Face
34711                                                                      LAZY BOY Vs. TRAMPOLINE from 45m!
31571                                         Is LeBron James as clutch as Michael Jordan? | The Jump | ESPN
168                           Fire breathing full twist backflips in one minute - Guinness World Records Day
34864                                          LeBron James' Dominant Performance & Buzzer Beater vs Toronto
39083                                  Brad Makes Whole Hog BBQ with Rodney Scott | It's Alive | Bon Appétit
22082                                               The Voice 2018 Blind Audition - Justin Kilgore: Tomorrow
3397                                                       Binging with Babish: Rachel's Trifle from Friends
34607           Inf

In [29]:
def content_type(title):
    t = str(title).lower()
    rules = [
        ("Highlights", ["highlight", "goals", "recap", " vs ", " vs.", "nba", "nfl", "fifa", "wwe", "football", "soccer", "match", "game "]),
        ("Trailer / Teaser", ["trailer", "teaser", "first look", "movie", "film", "official clip"]),
        ("Music Video", ["official video", "music video", "lyrics", "song", "official audio", "feat.", " ft.", "ft "]),
        ("Interview", ["interview", "press conference", "talks", "reacts", "reaction", "responds", "speaks"]),
        ("Challenge / Comedy", ["challenge", "prank", "funny", "comedy", "sketch", "parody"]),
        ("Talk Show / Clip", ["show", "monologue", "clip", "tonight", "late show", "kimmel", "fallon", "colbert", "snl", "saturday night live", "ellen", "corden"]),
        ("Behind the Scenes", ["behind the scenes", "making of", "vlog", "bts"]),
    ]
    for name, words in rules:
        if any(w in t for w in words):
            return name
    return "Other"

df["content_type"] = df["title"].apply(content_type)
print(df["content_type"].value_counts())
print("Other share:", round((df["content_type"] == "Other").mean() * 100, 1), "%")

content_type
Other                 1317
Highlights             241
Trailer / Teaser       218
Talk Show / Clip       122
Music Video             57
Interview               50
Challenge / Comedy      43
Behind the Scenes       25
Name: count, dtype: int64
Other share: 63.5 %


In [30]:
out_cols = ["video_id", "title", "channel_title", "category", "content_type",
            "publish_time", "post_date", "post_day", "post_hour", "views", "likes",
            "comment_count", "engagement_rate", "is_deleted", "is_zero_views", "is_outlier"]
df[out_cols].to_csv("../data/clean/posts_clean.csv", index=False)
print("Saved", len(df), "rows")

Saved 2073 rows


In [31]:
import re

def content_type(title):
    t = str(title).lower()
    rules = [
        ("Talent / Competition Show", [r"the voice", r"audition", r"\bbgt\b", r"got talent", r"\bagt\b",
                                       r"idol", r"golden buzzer", r"blind audition", r"x factor"]),
        ("Sports Highlights", [r"highlight", r"\bgoals?\b", r"buzzer", r"top 10", r"best moments", r"\bvs\.?\b",
                               r"\bnba\b", r"\bnfl\b", r"\bufc\b", r"\bwwe\b", r"wrestlemania", r"\bfifa\b",
                               r"world cup", r"\bmatch\b", r"performance"]),
        ("Sports Debate / Analysis", [r"first take", r"the jump", r"\bespn\b", r"preview", r"\breacts?\b", r"analysis"]),
        ("Trailer / Promo", [r"trailer", r"teaser", r"first look", r"announcement", r"\bnetflix\b",
                             r"avengers", r"marvel", r"\bmovie\b", r"\bfilm\b", r"season \d", r"episode",
                             r"official clip", r"justice league"]),
        ("Music", [r"official video", r"music video", r"lyrics", r"\bsong\b", r"official audio",
                   r"\bfeat\b", r"\bft\b", r"\blive\b"]),
        ("Food / Lifestyle", [r"\bbbq\b", r"babish", r"epicurious", r"bon app", r"recipe", r"cook",
                              r"it's alive", r"\beat\b", r"\bfood\b"]),
        ("Challenge / Stunt / Comedy", [r"challenge", r"prank", r"funny", r"comedy", r"sketch", r"parody",
                                        r"world record", r"guinness", r"parkour", r"i dare you", r"\btry\b"]),
        ("Interview / Talk Show", [r"interview", r"press conference", r"conversations", r"explains", r"\btalks?\b",
                                   r"tonight", r"late show", r"kimmel", r"fallon", r"colbert", r"\bsnl\b",
                                   r"saturday night live", r"ellen", r"corden", r"monologue", r"\bshow\b"]),
        ("Behind the Scenes / Vlog", [r"behind the scenes", r"making of", r"vlog", r"\bbts\b"]),
    ]
    for name, patterns in rules:
        if any(re.search(p, t) for p in patterns):
            return name
    return "Other"

df["content_type"] = df["title"].apply(content_type)
print(df["content_type"].value_counts())
print("Other share:", round((df["content_type"] == "Other").mean() * 100, 1), "%")

content_type
Other                         1076
Trailer / Promo                279
Sports Highlights              238
Interview / Talk Show          138
Sports Debate / Analysis       111
Music                           76
Talent / Competition Show       57
Challenge / Stunt / Comedy      48
Food / Lifestyle                29
Behind the Scenes / Vlog        21
Name: count, dtype: int64
Other share: 51.9 %


In [32]:
df["content_type"] = (df["title"].astype(str) + " " + df["channel_title"].astype(str)).apply(content_type)
print(df["content_type"].value_counts())
print("Other share:", round((df["content_type"] == "Other").mean() * 100, 1), "%")

content_type
Other                         848
Sports Highlights             282
Trailer / Promo               281
Interview / Talk Show         229
Music                         165
Sports Debate / Analysis      113
Talent / Competition Show      57
Challenge / Stunt / Comedy     46
Food / Lifestyle               29
Behind the Scenes / Vlog       23
Name: count, dtype: int64
Other share: 40.9 %


In [33]:
out_cols = ["video_id", "title", "channel_title", "category", "content_type",
            "publish_time", "post_date", "post_day", "post_hour", "views", "likes",
            "comment_count", "engagement_rate", "is_deleted", "is_zero_views", "is_outlier"]
df[out_cols].to_csv("../data/clean/posts_clean.csv", index=False)
print("Saved", len(df), "rows")

Saved 2073 rows


In [34]:
test = pd.DataFrame([
    {"video_id": "TEST_ZERO", "title": "Test zero views", "channel_title": "Test",
     "category": "Sports", "content_type": "Other", "publish_time": "2018-01-01 10:00:00+00:00",
     "post_date": "2018-01-01", "post_day": "Monday", "post_hour": 15,
     "views": 0, "likes": 0, "comment_count": 0},
    {"video_id": "TEST_NORMAL", "title": "Test normal video", "channel_title": "Test",
     "category": "Entertainment", "content_type": "Other", "publish_time": "2018-01-02 10:00:00+00:00",
     "post_date": "2018-01-02", "post_day": "Tuesday", "post_hour": 15,
     "views": 1000, "likes": 50, "comment_count": 10},
])
test["is_deleted"] = False
test["is_zero_views"] = test["views"] == 0
test["is_outlier"] = False
test["engagement_rate"] = np.where(test["views"] > 0,
    (test["likes"] + test["comment_count"]) / test["views"].replace(0, np.nan) * 100, np.nan)
test.to_csv("../data/clean/test_cases.csv", index=False)
test[["video_id", "views", "engagement_rate"]]

,video_id,views,engagement_rate
0,TEST_ZERO,0,NaN
1,TEST_NORMAL,1000,6.0
